In [31]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [32]:
df = pd.read_json('book_engagement_metrics.json', orient='records', lines=True)

In [33]:
print(df.columns)

Index(['id', 'like_count', 'region_code', 'video_description',
       'favorites_count', 'hashtag_names', 'effect_ids', 'is_stem_verified',
       'playlist_id', 'username', 'video_duration', 'video_label',
       'comment_count', 'create_time', 'music_id', 'view_count',
       'video_mention_list', 'voice_to_text', 'hashtag_info_list',
       'share_count', 'sticker_info_list', 'effect_info_list', 'video_tag',
       'full_text', 'book_title', 'book_category', 'view_count_pct_rank',
       'like_count_pct_rank', 'share_count_pct_rank',
       'favorites_count_pct_rank', 'comment_count_pct_rank',
       'raw_engagement_metrics', 'engagement_metrics'],
      dtype='object')


# 0.Twórcy

In [34]:
df['creator_video_count'] = df.groupby('username')['username'].transform('count')
df['creator_video_count']

0         6
1         4
2         1
3        41
4        69
         ..
51265    11
51266    22
51267    15
51268     4
51269    24
Name: creator_video_count, Length: 51270, dtype: int64

In [35]:
df['creator_avg_views'] = df.groupby('username')['view_count'].transform('mean')
df['creator_avg_views']

0         3850.833333
1          944.750000
2         3659.000000
3        10596.268293
4         1019.942029
             ...     
51265      945.727273
51266     4609.363636
51267     1778.600000
51268     1387.500000
51269      540.333333
Name: creator_avg_views, Length: 51270, dtype: float64

# 1. Identifying trending sounds (TOP 5% by frequency)

In [36]:
# 1. Identifying trending sounds (TOP 5% by frequency)
top_audio = df['music_id'].value_counts().nlargest(int(len(df['music_id'].unique()) * 0.05)).index
print("Trending Sounds (Top 5% by frequency) with their counts:")
for audio in top_audio:
    count = df[df['music_id'] == audio].shape[0]
    print(f"{audio}: {count}")

Trending Sounds (Top 5% by frequency) with their counts:
7547451235554265886: 86
0: 85
7089108976524888066: 61
7463260542765976352: 52
7537258969614666529: 51
7536694526191127327: 51
7412332770956249889: 49
7455003793273457450: 46
7343015239631244078: 44
7253916111752907546: 44
6985759482454215430: 42
7449067587487058718: 42
7530439116829313040: 41
7128512071482493701: 40
7412027569211869995: 40
7456817705721301766: 40
7438474281459272494: 39
7410878726026496016: 39
7466220526407535403: 38
7494705068190747447: 37
7290790746045631234: 35
218452606472302592: 35
7537026735125793558: 34
7269085456229878560: 34
7270043916493637633: 34
7188940380900657926: 33
7545958751400954654: 33
7282498671971780609: 32
7438050909496265515: 32
7445702424729275168: 31
7118336785461119749: 30
6823684463831025665: 30
7474209817062574850: 30
7362791883656940289: 29
7503500280685643792: 29
7451861729821444097: 29
7528919621370399502: 28
7500659531221027606: 28
7421455946192668673: 28
6817665487665629186: 28
74

In [37]:
df['is_trending_music'] = df['music_id'].isin(top_audio).astype(int)
print(df['is_trending_music'].value_counts())

is_trending_music
0    37007
1    14263
Name: count, dtype: int64


In [38]:
df['is_trending_music']

0        1
1        1
2        0
3        1
4        0
        ..
51265    0
51266    1
51267    0
51268    0
51269    0
Name: is_trending_music, Length: 51270, dtype: int64

# 2. Using Effects and Stickers

In [39]:
# 2. Using Effects and Stickers
df['has_effects'] = df['effect_ids'].apply(lambda x: 1 if isinstance(x, list) and len(x) > 0 else 0)
df['effects_count'] = df['effect_ids'].apply(lambda x: len(x) if isinstance(x, list) else 0)
df['has_stickers'] = df['sticker_info_list'].apply(lambda x: 1 if isinstance(x, list) and len(x) > 0 else 0)


In [40]:
print(df['has_effects'].value_counts())
print(df['has_stickers'].value_counts())
print(df['effects_count'].describe())

has_effects
0    45198
1     6072
Name: count, dtype: int64
has_stickers
1    29334
0    21936
Name: count, dtype: int64
count    51270.000000
mean         0.122157
std          0.339691
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          5.000000
Name: effects_count, dtype: float64


# 3. Categorizing Video Lengths


In [41]:
print(df['video_duration'].describe())

count    51270.000000
mean        22.735713
std         58.565714
min          0.000000
25%          0.000000
50%         11.000000
75%         17.000000
max       2319.000000
Name: video_duration, dtype: float64


In [42]:
bins = [0, 15, 60, np.inf]
labels = ['Short (<15s)', 'Medium (15-60s)', 'Long (>60s)']
df['duration_category'] = pd.cut(df['video_duration'], bins=bins, labels=labels)


# 4. Hashtag Strategy Analysis

In [43]:
# 4. Hashtag Strategy Analysis
df['hashtag_count'] = df['hashtag_names'].apply(lambda x: len(x) if isinstance(x, list) else 0)

# Determining the presence of niche tags (e.g., genre tags)
niche_tags = {'darkacademia', 'romantasy', 'fantasy', 'darkromance', 'enemiestolovers', 'mafiaromance'}
df['has_niche_hashtag'] = df['hashtag_names'].apply(
    lambda x: 1 if isinstance(x, list) and any(tag.lower() in niche_tags for tag in x) else 0
)


In [44]:
print(df['has_niche_hashtag'].value_counts())

has_niche_hashtag
0    47246
1     4024
Name: count, dtype: int64


In [45]:
print(df['hashtag_count'].describe())

count    51270.000000
mean         8.427833
std          7.564672
min          1.000000
25%          5.000000
50%          5.000000
75%         10.000000
max        149.000000
Name: hashtag_count, dtype: float64


#  6. Preparing the X and Y matrices

In [46]:
# Preview of newly created variables
print(df[['is_trending_music', 'has_effects', 'has_stickers', 'duration_category', 'has_niche_hashtag', 'creator_video_count', 'creator_avg_views']].head())

   is_trending_music  has_effects  has_stickers duration_category  \
0                  1            0             0               NaN   
1                  1            0             1               NaN   
2                  0            1             1      Short (<15s)   
3                  1            0             0   Medium (15-60s)   
4                  0            0             0      Short (<15s)   

   has_niche_hashtag  creator_video_count  creator_avg_views  
0                  0                    6        3850.833333  
1                  0                    4         944.750000  
2                  0                    1        3659.000000  
3                  0                   41       10596.268293  
4                  0                   69        1019.942029  


In [49]:
# Converting `duration_category` to binary variables (encoding)
duration_dummies = pd.get_dummies(df['duration_category'], drop_first=True, dtype=int)
book_dummies = pd.get_dummies(df['book_title'], drop_first=True, prefix='book', dtype=int)
print(duration_dummies)
print(book_dummies)

       Medium (15-60s)  Long (>60s)
0                    0            0
1                    0            0
2                    0            0
3                    1            0
4                    0            0
...                ...          ...
51265                0            1
51266                0            0
51267                0            0
51268                0            0
51269                1            0

[51270 rows x 2 columns]
       book_Boys of Tommen  book_Co wyszeptał nam deszcz  book_Fourth Wing  \
0                        0                             0                 0   
1                        0                             0                 0   
2                        0                             0                 0   
3                        0                             0                 0   
4                        0                             0                 0   
...                    ...                           ...               ...

In [ ]:
# features of visual and auditory explanatory variables and their creators
X_features = [
    'is_trending_music', 
    'effects_count', 
    'has_stickers', 
    'hashtag_count', 
    'has_niche_hashtag',
    'creator_video_count',
    'creator_avg_views'
]